# Data Generation — Run Once

This notebook was used to generate the synthetic dataset 
for the Warehouse Analytics project.

**Do not re-run** — this will overwrite existing data in /data/raw/
and may cause inconsistencies with the SQL analysis results.

Dataset generated: 2023-01-01 to 2023-12-31
Records: ~45,000 across 9 tables

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os

In [2]:
fake = Faker('pl_PL')
np.random.seed(42)
random.seed(42)

os.makedirs('../data/raw', exist_ok=True)

print("Biblioteki załadowane ✓")

Biblioteki załadowane ✓


In [ ]:
# TABLE 1: SUPPLIERS
# 15 suppliers from different countries with varying reliability

countries = {
    'Polska':  {'lead_time': (2, 5),   'reliability': (92, 99)},
    'Niemcy':  {'lead_time': (3, 7),   'reliability': (90, 97)},
    'Chiny':   {'lead_time': (20, 35), 'reliability': (75, 88)},
    'Turcja':  {'lead_time': (7, 14),  'reliability': (82, 93)},
    'Włochy':  {'lead_time': (5, 10),  'reliability': (88, 96)},
}

supplier_names = [
    'TechSupply Sp. z o.o.', 'MegaDistrib GmbH', 'ShanghaiGoods Ltd.',
    'IstanbulTrade Co.', 'MilanExport S.r.l.', 'WarszawaDystrybucja',
    'BerlinLogistics AG', 'BeijingSource Ltd.', 'AnkaraSupply Co.',
    'RomaTrade S.r.l.', 'KrakówHurt Sp. z o.o.', 'MünchenWaren GmbH',
    'ShenzhenTech Ltd.', 'IzmirGoods Co.', 'FlorenceItems S.r.l.'
]

country_list = ['Polska', 'Niemcy', 'Chiny', 'Turcja', 'Włochy',
                'Polska', 'Niemcy', 'Chiny', 'Turcja', 'Włochy',
                'Polska', 'Niemcy', 'Chiny', 'Turcja', 'Włochy']

suppliers = []
for i, (name, country) in enumerate(zip(supplier_names, country_list), start=1):
    params = countries[country]
    suppliers.append({
        'supplier_id':     i,
        'supplier_name':   name,
        'country':         country,
        'lead_time_days':  random.randint(*params['lead_time']),
        'reliability_pct': round(random.uniform(*params['reliability']), 2),
        'created_at':      datetime(2023, 1, 1)
    })

df_suppliers = pd.DataFrame(suppliers)
df_suppliers.to_csv('../data/raw/suppliers.csv', index=False)

print(df_suppliers)
print(f"\nWygenerowano {len(df_suppliers)} dostawców ✓")

    supplier_id          supplier_name country  lead_time_days  \
0             1  TechSupply Sp. z o.o.  Polska               2   
1             2       MegaDistrib GmbH  Niemcy               5   
2             3     ShanghaiGoods Ltd.   Chiny              24   
3             4      IstanbulTrade Co.  Turcja               8   
4             5     MilanExport S.r.l.  Włochy               5   
5             6    WarszawaDystrybucja  Polska               3   
6             7     BerlinLogistics AG  Niemcy               7   
7             8     BeijingSource Ltd.   Chiny              26   
8             9       AnkaraSupply Co.  Turcja              13   
9            10       RomaTrade S.r.l.  Włochy               9   
10           11  KrakówHurt Sp. z o.o.  Polska               2   
11           12      MünchenWaren GmbH  Niemcy               4   
12           13      ShenzhenTech Ltd.   Chiny              30   
13           14         IzmirGoods Co.  Turcja              10   
14        

In [ ]:
# TABLE 2: SKUS
# 500 products across 5 categories
# ABC class: A=top products, B=medium, C=long tail

categories = {
    'Electronics':  {'price_range': (50, 800),  'cost_margin': 0.55, 'weight': (0.1, 2.5)},
    'Clothing':     {'price_range': (20, 200),  'cost_margin': 0.40, 'weight': (0.1, 0.8)},
    'Home & Garden':{'price_range': (15, 300),  'cost_margin': 0.45, 'weight': (0.2, 5.0)},
    'Sports':       {'price_range': (25, 400),  'cost_margin': 0.50, 'weight': (0.2, 3.0)},
    'Beauty':       {'price_range': (10, 150),  'cost_margin': 0.35, 'weight': (0.05, 0.5)},
}

# ABC class: 20% of products are A, 30% are B, 50% are C
abc_classes = ['A'] * 100 + ['B'] * 150 + ['C'] * 250
random.shuffle(abc_classes)

skus = []
for i in range(1, 501):
    category = random.choice(list(categories.keys()))
    params    = categories[category]
    
    unit_price = round(random.uniform(*params['price_range']), 2)
    unit_cost  = round(unit_price * params['cost_margin'], 2)
    
    skus.append({
        'sku_id':       i,
        'sku_code':     f'SKU-{i:04d}',
        'product_name': f'{category} Product {i:04d}',
        'category':     category,
        'unit_cost':    unit_cost,
        'unit_price':   unit_price,
        'weight_kg':    round(random.uniform(*params['weight']), 3),
        'abc_class':    abc_classes[i - 1],
        'supplier_id':  random.randint(1, 15),
        'created_at':   datetime(2023, 1, 1)
    })

df_skus = pd.DataFrame(skus)
df_skus.to_csv('../data/raw/skus.csv', index=False)

# Quick verification of the generated data
print(df_skus['abc_class'].value_counts())
print(f"\nRozkład kategorii:")
print(df_skus['category'].value_counts())
print(f"\nWygenerowano {len(df_skus)} SKU ✓")

abc_class
C    250
B    150
A    100
Name: count, dtype: int64

Rozkład kategorii:
category
Sports           108
Beauty           104
Home & Garden    100
Electronics       97
Clothing          91
Name: count, dtype: int64

Wygenerowano 500 SKU ✓


In [ ]:
# TABLE 3: EMPLOYEES
# 50 warehouse employees across 4 roles and 3 shifts

roles  = ['picker'] * 25 + ['receiver'] * 10 + ['packer'] * 10 + ['supervisor'] * 5
shifts = ['morning', 'afternoon', 'night']

employees = []
for i in range(1, 51):
    hire_date = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 180))
    
    employees.append({
        'employee_id':   i,
        'employee_code': f'EMP-{i:03d}',
        'first_name':    fake.first_name(),
        'last_name':     fake.last_name(),
        'role':          roles[i - 1],
        'shift':         random.choice(shifts),
        'hire_date':     hire_date.date(),
        'is_active':     random.choices([True, False], weights=[90, 10])[0],
        'created_at':    datetime(2023, 1, 1)
    })

df_employees = pd.DataFrame(employees)
df_employees.to_csv('../data/raw/employees.csv', index=False)

print(df_employees[['employee_id', 'employee_code', 'first_name', 
                     'last_name', 'role', 'shift']].head(10))
print(f"\nRozkład ról:")
print(df_employees['role'].value_counts())
print(f"\nWygenerowano {len(df_employees)} pracowników ✓")

   employee_id employee_code first_name    last_name    role      shift
0            1       EMP-001      Klara        Koczy  picker    morning
1            2       EMP-002    Mikołaj       Budzeń  picker    morning
2            3       EMP-003      Filip      Bojczuk  picker      night
3            4       EMP-004     Julian   Wincenciak  picker  afternoon
4            5       EMP-005    Malwina        Zugaj  picker  afternoon
5            6       EMP-006    Aurelia      Flisiak  picker      night
6            7       EMP-007   Krystian          Cop  picker    morning
7            8       EMP-008     Damian          Paw  picker  afternoon
8            9       EMP-009     Kacper  Ryszkiewicz  picker  afternoon
9           10       EMP-010  Krzysztof      Dziurla  picker    morning

Rozkład ról:
role
picker        25
receiver      10
packer        10
supervisor     5
Name: count, dtype: int64

Wygenerowano 50 pracowników ✓


In [ ]:
# TABLE 4: PURCHASE_ORDERS
# ~1200 purchase orders to suppliers over 12 months

start_date = datetime(2023, 1, 1)
end_date   = datetime(2023, 12, 31)

# Supplier dictionary - we need lead_time and reliability per supplier
suppliers_dict = df_suppliers.set_index('supplier_id')[
    ['lead_time_days', 'reliability_pct']
].to_dict('index')

purchase_orders = []
po_id = 1

for _ in range(1200):
    supplier_id = random.randint(1, 15)
    sku_id      = random.randint(1, 500)
    supplier    = suppliers_dict[supplier_id]
    
    order_date    = start_date + timedelta(days=random.randint(0, 364))
    expected_date = order_date + timedelta(days=supplier['lead_time_days'])
    
    # Will the supplier deliver on time?
    # reliability_pct = 85 means an 85% chance of on-time delivery
    is_reliable = random.random() < (supplier['reliability_pct'] / 100)
    
    if is_reliable:
        # Delivery on time or 1-2 days early
        delay_days = random.randint(-2, 1)
        status     = 'received'
    else:
        # Delay of 3-21 days
        delay_days = random.randint(3, 21)
        status     = 'received'
    
    actual_date = expected_date + timedelta(days=delay_days)
    
    sku_data   = df_skus[df_skus['sku_id'] == sku_id].iloc[0]
    ordered_qty = random.randint(50, 500)
    
    purchase_orders.append({
        'po_id':        po_id,
        'supplier_id':  supplier_id,
        'sku_id':       sku_id,
        'ordered_qty':  ordered_qty,
        'unit_cost':    sku_data['unit_cost'],
        'status':       status,
        'order_date':   order_date,
        'expected_date':expected_date,
        'actual_date':  actual_date,
        'created_at':   order_date
    })
    po_id += 1

df_po = pd.DataFrame(purchase_orders)
df_po.to_csv('../data/raw/purchase_orders.csv', index=False)

print(df_po[['po_id','supplier_id','sku_id','ordered_qty',
             'order_date','expected_date','actual_date']].head(5))
print(f"\nWygenerowano {len(df_po)} purchase orders ✓")

   po_id  supplier_id  sku_id  ordered_qty order_date expected_date  \
0      1            1     474          328 2023-02-08    2023-02-10   
1      2           10     337           58 2023-01-17    2023-01-26   
2      3            4     301           89 2023-03-15    2023-03-23   
3      4            5      84          396 2023-10-16    2023-10-21   
4      5           15     277          454 2023-06-19    2023-06-26   

  actual_date  
0  2023-02-10  
1  2023-01-26  
2  2023-03-24  
3  2023-10-22  
4  2023-06-25  

Wygenerowano 1200 purchase orders ✓


In [7]:
df_po['delay_days'] = (df_po['actual_date'] - df_po['expected_date']).dt.days
print("Opóźnienia (dni):")
print(df_po['delay_days'].describe().round(2))
print(f"\nOpóźnionych: {(df_po['delay_days'] > 0).sum()} z {len(df_po)}")
print(f"Na czas lub wcześniej: {(df_po['delay_days'] <= 0).sum()}")

Opóźnienia (dni):
count    1200.00
mean        1.06
std         4.65
min        -2.00
25%        -1.00
50%         0.00
75%         1.00
max        21.00
Name: delay_days, dtype: float64

Opóźnionych: 428 z 1200
Na czas lub wcześniej: 772


In [ ]:
# TABLE 5: RECEIVING_LOG
# Each PO generates one receiving log entry
# received_qty may be less than ordered_qty

# Only active receivers handle receipts
receivers = df_employees[df_employees['role'] == 'receiver']['employee_id'].tolist()

receiving_log = []

for _, po in df_po.iterrows():
    
    # Did the supplier deliver the full order?
    # 80% chance of full delivery, 20% partial
    full_delivery = random.random() < 0.80
    
    if full_delivery:
        received_qty = int(po['ordered_qty'])
    else:
        # Partial delivery - 60% to 99% of ordered quantity
        pct          = random.uniform(0.60, 0.99)
        received_qty = int(po['ordered_qty'] * pct)
    
    # Damaged goods - 0% to 3% of received quantity
    damaged_qty = int(received_qty * random.uniform(0, 0.03))
    
    # Putaway - shelving takes 1 to 8 hours after receipt
    received_at = po['actual_date'] + timedelta(hours=random.randint(1, 4))
    putaway_at  = received_at + timedelta(hours=random.randint(1, 8))
    
    # Warehouse location: row A-E, rack 01-20, shelf 01-05
    row      = random.choice(['A', 'B', 'C', 'D', 'E'])
    rack     = random.randint(1, 20)
    shelf    = random.randint(1, 5)
    location = f'{row}-{rack:02d}-{shelf:02d}'
    
    receiving_log.append({
        'receiving_id':  len(receiving_log) + 1,
        'po_id':         int(po['po_id']),
        'sku_id':        int(po['sku_id']),
        'employee_id':   random.choice(receivers),
        'received_qty':  received_qty,
        'damaged_qty':   damaged_qty,
        'received_at':   received_at,
        'putaway_at':    putaway_at,
        'location_code': location,
        'notes':         None,
        'created_at':    received_at
    })

df_receiving = pd.DataFrame(receiving_log)
df_receiving.to_csv('../data/raw/receiving_log.csv', index=False)

print(df_receiving[['receiving_id', 'po_id', 'sku_id', 
                     'received_qty', 'damaged_qty', 
                     'received_at', 'putaway_at', 
                     'location_code']].head(5))

# Verification of receiving log
full = (df_receiving['received_qty'] == df_po['ordered_qty']).sum()
print(f"\nPełne dostawy:    {full} ({full/len(df_receiving)*100:.1f}%)")
print(f"Częściowe dostawy: {len(df_receiving)-full} ({(len(df_receiving)-full)/len(df_receiving)*100:.1f}%)")
print(f"Średnio uszkodzonych: {df_receiving['damaged_qty'].mean():.1f} szt. per dostawa")
print(f"\nWygenerowano {len(df_receiving)} wpisów receiving_log ✓")

   receiving_id  po_id  sku_id  received_qty  damaged_qty         received_at  \
0             1      1     474           328            3 2023-02-10 01:00:00   
1             2      2     337            58            0 2023-01-26 01:00:00   
2             3      3     301            89            2 2023-03-24 03:00:00   
3             4      4      84           364            5 2023-10-22 02:00:00   
4             5      5     277           454            7 2023-06-25 01:00:00   

           putaway_at location_code  
0 2023-02-10 05:00:00       D-03-02  
1 2023-01-26 02:00:00       D-14-01  
2 2023-03-24 07:00:00       A-16-02  
3 2023-10-22 04:00:00       E-20-04  
4 2023-06-25 02:00:00       A-13-03  

Pełne dostawy:    933 (77.8%)
Częściowe dostawy: 267 (22.2%)
Średnio uszkodzonych: 3.4 szt. per dostawa

Wygenerowano 1200 wpisów receiving_log ✓


In [ ]:
# TABLE 6: INVENTORY
# Current inventory status - one record per SKU
# Calculated based on what we received (receiving_log)

# Sum received_qty per SKU from receiving_log
received_per_sku = (
    df_receiving
    .groupby('sku_id')['received_qty']
    .sum()
    .reset_index()
    .rename(columns={'received_qty': 'total_received'})
)

inventory = []

for _, row in received_per_sku.iterrows():
    on_hand  = int(row['total_received'])
    
    # Some inventory is already reserved for customer orders
    # 10% to 30% of on-hand stock
    reserved = int(on_hand * random.uniform(0.10, 0.30))
    
    # Reorder point - when we reorder new stock
    # Depends on ABC class: A-products have a higher reorder point
    sku_class = df_skus[df_skus['sku_id'] == row['sku_id']]['abc_class'].values[0]
    
    reorder = {'A': random.randint(100, 200),
               'B': random.randint(50, 100),
               'C': random.randint(10, 50)}[sku_class]
    
    inventory.append({
        'inventory_id':       len(inventory) + 1,
        'sku_id':             int(row['sku_id']),
        'quantity_on_hand':   on_hand,
        'quantity_reserved':  reserved,
        'quantity_available': on_hand - reserved,
        'reorder_point':      reorder,
        'last_updated':       datetime(2023, 12, 31)
    })

df_inventory = pd.DataFrame(inventory)
df_inventory.to_csv('../data/raw/inventory.csv', index=False)

print(df_inventory.head(5))
print(f"\nSKU poniżej reorder point (potrzebują zamówienia):")
below = df_inventory[df_inventory['quantity_available'] < df_inventory['reorder_point']]
print(f"{len(below)} SKU z {len(df_inventory)} ({len(below)/len(df_inventory)*100:.1f}%)")
print(f"\nWygenerowano {len(df_inventory)} wpisów inventory ✓")

   inventory_id  sku_id  quantity_on_hand  quantity_reserved  \
0             1       1               886                132   
1             2       3               819                 86   
2             3       4               337                 43   
3             4       5              1070                227   
4             5       7              1235                162   

   quantity_available  reorder_point last_updated  
0                 754             89   2023-12-31  
1                 733             82   2023-12-31  
2                 294             19   2023-12-31  
3                 843            113   2023-12-31  
4                1073             25   2023-12-31  

SKU poniżej reorder point (potrzebują zamówienia):
9 SKU z 448 (2.0%)

Wygenerowano 448 wpisów inventory ✓


In [ ]:
# TABLE 7: CUSTOMER_ORDERS
# 15,000 customer orders over 12 months
# Seasonality: more orders in Q4 (holidays, Black Friday)

channels = ['website', 'mobile_app', 'marketplace']
statuses = ['shipped', 'delivered', 'cancelled', 'returned']

# Monthly weights - Q4 has higher volume
monthly_weights = {
    1: 0.06, 2: 0.06, 3: 0.07, 4: 0.07, 5: 0.08, 6: 0.08,
    7: 0.08, 8: 0.07, 9: 0.08, 10: 0.09, 11: 0.12, 12: 0.14
}

# Generate dates with seasonality
all_dates = []
for month, weight in monthly_weights.items():
    n_orders = int(15000 * weight)
    for _ in range(n_orders):
        day = random.randint(1, 28)
        all_dates.append(datetime(2023, month, day,
                                  random.randint(8, 22),
                                  random.randint(0, 59)))

random.shuffle(all_dates)

customer_orders = []

for i, order_date in enumerate(all_dates[:15000], start=1):
    sku        = df_skus.sample(1, weights=df_skus['abc_class'].map(
                     {'A': 0.70, 'B': 0.20, 'C': 0.10})).iloc[0]
    quantity   = random.randint(1, 5)
    unit_price = sku['unit_price']
    
    # Order status
    status = random.choices(
        statuses,
        weights=[0.15, 0.75, 0.05, 0.05]
    )[0]

    customer_orders.append({
        'order_id':     i,
        'order_code':   f'ORD-{order_date.strftime("%Y%m%d")}-{i:05d}',
        'customer_id':  random.randint(1, 5000),
        'sku_id':       int(sku['sku_id']),
        'quantity':     quantity,
        'unit_price':   unit_price,
        'total_value':  round(unit_price * quantity, 2),
        'channel':      random.choices(channels, weights=[0.50, 0.35, 0.15])[0],
        'status':       status,
        'order_date':   order_date,
        'required_date':order_date + timedelta(days=random.randint(2, 7)),
        'created_at':   order_date
    })

df_co = pd.DataFrame(customer_orders)
df_co.to_csv('../data/raw/customer_orders.csv', index=False)

print(df_co[['order_id','order_code','sku_id',
             'quantity','total_value','channel','status']].head(5))
print(f"\nRozkład statusów:")
print(df_co['status'].value_counts())
print(f"\nRozkład kanałów:")
print(df_co['channel'].value_counts())
print(f"\nWygenerowano {len(df_co)} customer orders ✓")

   order_id          order_code  sku_id  quantity  total_value     channel  \
0         1  ORD-20230820-00001     189         5       216.60  mobile_app   
1         2  ORD-20230623-00002     475         4       278.48  mobile_app   
2         3  ORD-20230626-00003     362         4      1125.60     website   
3         4  ORD-20231125-00004     298         2       334.02  mobile_app   
4         5  ORD-20231023-00005      77         2       108.96  mobile_app   

      status  
0  delivered  
1  delivered  
2  delivered  
3  delivered  
4  delivered  

Rozkład statusów:
status
delivered    11232
shipped       2294
returned       759
cancelled      715
Name: count, dtype: int64

Rozkład kanałów:
channel
website        7523
mobile_app     5210
marketplace    2267
Name: count, dtype: int64

Wygenerowano 15000 customer orders ✓


In [ ]:
# TABLE 8: PICK_ORDERS
# One customer_order = one pick_order
# Only orders not cancelled generate picking

pickers     = df_employees[df_employees['role'] == 'picker']['employee_id'].tolist()
error_types = ['wrong_item', 'wrong_qty', 'damaged']

# Only orders that went into fulfillment
orders_to_pick = df_co[df_co['status'] != 'cancelled'].copy()

pick_orders = []

for _, order in orders_to_pick.iterrows():
    
    # Picking start time - 1 to 4 hours after order placement
    started_at   = order['order_date'] + timedelta(hours=random.randint(1, 4),
                                                    minutes=random.randint(0, 59))
    
    # Completion time - 5 to 30 minutes
    completed_at = started_at + timedelta(minutes=random.randint(5, 30))
    
    # Pick accuracy - 98% correct picks
    is_correct = random.random() < 0.98
    
    if is_correct:
        qty_picked = int(order['quantity'])
        error_type = None
    else:
        qty_picked = int(order['quantity']) - random.randint(1, int(order['quantity']))
        error_type = random.choice(error_types)

    pick_orders.append({
        'pick_id':      len(pick_orders) + 1,
        'order_id':     int(order['order_id']),
        'employee_id':  random.choice(pickers),
        'sku_id':       int(order['sku_id']),
        'qty_requested':int(order['quantity']),
        'qty_picked':   qty_picked,
        'is_correct':   is_correct,
        'error_type':   error_type,
        'started_at':   started_at,
        'completed_at': completed_at,
        'lines_picked': 1,
        'created_at':   started_at
    })

df_picks = pd.DataFrame(pick_orders)
df_picks.to_csv('../data/raw/pick_orders.csv', index=False)

print(df_picks[['pick_id','order_id','employee_id',
                'qty_requested','qty_picked',
                'is_correct','error_type']].head(5))

errors = df_picks[df_picks['is_correct'] == False]
print(f"\nPick Accuracy: {(1 - len(errors)/len(df_picks))*100:.2f}%")
print(f"Błędy per typ:")
print(errors['error_type'].value_counts())
print(f"\nWygenerowano {len(df_picks)} pick orders ✓")

   pick_id  order_id  employee_id  qty_requested  qty_picked  is_correct  \
0        1         1           14              5           5        True   
1        2         2           24              4           4        True   
2        3         3           12              4           4        True   
3        4         4            7              2           2        True   
4        5         5           21              2           2        True   

  error_type  
0       None  
1       None  
2       None  
3       None  
4       None  

Pick Accuracy: 97.85%
Błędy per typ:
error_type
wrong_item    116
damaged        96
wrong_qty      95
Name: count, dtype: int64

Wygenerowano 14285 pick orders ✓


In [ ]:
# TABLE 9: SHIPMENTS
# Only shipped and delivered orders generate shipments
# SLA: we promise customer delivery in 2-3 business days

carriers = ['DPD', 'DHL', 'InPost', 'UPS']

# Only orders that left the warehouse
orders_to_ship = df_co[df_co['status'].isin(['shipped', 'delivered'])].copy()

shipments = []

for _, order in orders_to_ship.iterrows():
    
    carrier = random.choices(
        carriers,
        weights=[0.35, 0.25, 0.30, 0.10]
    )[0]
    
    # Shipping cost depends on the carrier
    cost_ranges = {
        'DPD':   (8, 18),
        'DHL':   (12, 25),
        'InPost': (6, 14),
        'UPS':   (15, 35)
    }
    shipping_cost = round(random.uniform(*cost_ranges[carrier]), 2)
    
    # Shipment 2-8 hours after picking completion
    pick_data  = df_picks[df_picks['order_id'] == order['order_id']]
    if len(pick_data) == 0:
        continue
    
    ship_date      = pick_data.iloc[0]['completed_at'] + timedelta(hours=random.randint(2, 8))
    sla_days       = random.randint(2, 3)
    estimated_date = ship_date + timedelta(days=sla_days)
    
    # Was it delivered on time?
    # DHL and UPS are more reliable, InPost and DPD less so
    on_time_prob = {
        'DPD':    0.88,
        'DHL':    0.94,
        'InPost': 0.91,
        'UPS':    0.95
    }[carrier]
    
    is_on_time = random.random() < on_time_prob
    
    if is_on_time:
        actual_date = estimated_date - timedelta(hours=random.randint(0, 12))
    else:
        actual_date = estimated_date + timedelta(days=random.randint(1, 5))
    
    # Package weight
    sku_weight  = df_skus[df_skus['sku_id'] == order['sku_id']]['weight_kg'].values[0]
    total_weight = round(sku_weight * order['quantity'] + 0.5, 3)
    
    shipments.append({
        'shipment_id':    len(shipments) + 1,
        'order_id':       int(order['order_id']),
        'carrier':        carrier,
        'tracking_number':f'{carrier}-{random.randint(100000000, 999999999)}',
        'shipping_cost':  shipping_cost,
        'weight_kg':      total_weight,
        'ship_date':      ship_date,
        'estimated_date': estimated_date,
        'actual_date':    actual_date,
        'sla_days':       sla_days,
        'is_on_time':     is_on_time,
        'created_at':     ship_date
    })

df_shipments = pd.DataFrame(shipments)
df_shipments.to_csv('../data/raw/shipments.csv', index=False)

print(df_shipments[['shipment_id','order_id','carrier',
                     'shipping_cost','is_on_time']].head(5))
print(f"\nOn-Time Rate per przewoźnik:")
print(df_shipments.groupby('carrier')['is_on_time'].mean().round(3) * 100)
print(f"\nŚredni koszt per przewoźnik:")
print(df_shipments.groupby('carrier')['shipping_cost'].mean().round(2))
print(f"\nWygenerowano {len(df_shipments)} shipments ✓")

   shipment_id  order_id carrier  shipping_cost  is_on_time
0            1         1     UPS          21.29        True
1            2         2     DPD          16.56        True
2            3         3     DPD          14.53        True
3            4         4     DPD          17.33        True
4            5         5  InPost          13.88        True

On-Time Rate per przewoźnik:
carrier
DHL       93.2
DPD       88.1
InPost    90.8
UPS       94.8
Name: is_on_time, dtype: float64

Średni koszt per przewoźnik:
carrier
DHL       18.51
DPD       12.95
InPost    10.01
UPS       24.95
Name: shipping_cost, dtype: float64

Wygenerowano 13526 shipments ✓
